# Inner-Boundary Magnetic Field Map (ZDI-Style Starter)

This example loads the 3D BATSRUS sample, samples the **inner boundary shell** (`R=1`) using the library, and makes lat/lon magnetic maps.

The notebook uses library helpers for **sampling** and **spherical component transforms**, then plots directly with Matplotlib (`pcolormesh`, `contour`, `quiver`).


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.data.samples import get_sample
from starwinds_analysis.analysis.shells import integrate_shell_scalar, sample_spherical_shells
from starwinds_analysis.recipes.spherical import spherical_vector_components

plt.rcParams['figure.dpi'] = 120


## Load the 3D Sample and Configure the Radius Scale

We feed the stellar radius into the BATSRUS/griblet graph near the top so SI conversions are available consistently.

In [ ]:
DATA_FILE = Path(get_sample('3d__var_1_n00060000.plt'))
STAR_RADIUS_M = 696_000_000.0

print('Using:', DATA_FILE)
sds = SmartDs.from_file(DATA_FILE)
sds.add_batsrus_graph(body_radius_m=STAR_RADIUS_M)
body_radius_m = STAR_RADIUS_M


# DONE do you just delete tehse todos or what? I told you to use print(sds).
print(sds)
print(f'Body radius [m]: {body_radius_m:.6e}')


## Sample the Inner Boundary Shell (`R = 1`) and Compute Components

This samples Cartesian magnetic field components on a regular spherical grid and computes:

- `B_r` (radial)
- `B_phi` (azimuthal / eastward)
- `B_meridional = -B_theta` (northward tangent)
- `|B_tan|`

Internal field values are SI (`T`); plotting below uses `G`.


In [ ]:
R_BOUNDARY = 1.0
N_POLAR = 64
N_AZIMUTH = 128
B_PLOT_UNIT = 'G'
B_SCALE = 1e4 if B_PLOT_UNIT == 'G' else 1.0

# DONE what is this shite function? You are doing it exactly the wrong way.
shell = sample_spherical_shells(
    sds,
    [R_BOUNDARY],
    fields=('B_x [T]', 'B_y [T]', 'B_z [T]'),
    n_polar=N_POLAR,
    n_azimuth=N_AZIMUTH,
    method='nearest',
    length_unit_to_m=body_radius_m,
)

bx = shell.fields['B_x [T]'][0]
by = shell.fields['B_y [T]'][0]
bz = shell.fields['B_z [T]'][0]
x = shell.x[0]
y = shell.y[0]
z = shell.z[0]

b_r_T, b_theta_T, b_phi_T = spherical_vector_components(bx, by, bz, x, y, z)
b_meridional_T = -b_theta_T  # northward on a latitude map
b_tan_T = np.sqrt(b_phi_T**2 + b_meridional_T**2)

lon_deg = np.degrees(shell.phi)
lat_deg = 90.0 - np.degrees(shell.theta)

b_r = B_SCALE * b_r_T
b_phi = B_SCALE * b_phi_T
b_meridional = B_SCALE * b_meridional_T
b_tan = B_SCALE * b_tan_T

print('Shell grid shape (theta, phi):', shell.theta.shape)
print('Radius:', float(shell.radii[0]))
print('Plot unit:', B_PLOT_UNIT)


## ZDI-Style Component Maps (Radial / Azimuthal / Meridional)

Conventions used here:

- **Radial**: outward normal component (`B_r`)
- **Azimuthal**: `+phi` direction (eastward on the lat/lon map)
- **Meridional**: northward tangent component (`-B_theta`) so it follows latitude intuition

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 10), sharex=True, constrained_layout=True)
components = [
    ('Radial', b_r),
    ('Azimuthal', b_phi),
    ('Meridional', b_meridional),
]
vabs = np.nanmax(np.abs(np.concatenate([arr.ravel() for _label, arr in components])))

for ax, (label, arr) in zip(np.ravel(axs), components):
    img = ax.pcolormesh(
        lon_deg,
        lat_deg,
        arr,
        shading='nearest',
        cmap='RdBu_r',
        vmin=-vabs,
        vmax=vabs,
    )
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.xaxis.set_major_locator(MultipleLocator(90))
    ax.xaxis.set_minor_locator(MultipleLocator(30))
    ax.yaxis.set_major_locator(MultipleLocator(45))
    ax.yaxis.set_minor_locator(MultipleLocator(15))
    ax.grid(which='major', alpha=0.15, linewidth=0.5)
    ax.set_ylabel('Latitude [deg]')
    ax.set_title(f'{label} field at R={R_BOUNDARY:g}')
    fig.colorbar(img, ax=ax, label=f'{label} [{B_PLOT_UNIT}]')

axs[-1].set_xlabel('Longitude [deg]')
plt.show()


## Tangential Field as Vectors on a Lat/Lon Map

This plot shows the tangential field in two ways:

- **Background color**: tangential magnetic strength `|B_tan|`
- **Arrows**: tangential field direction in the local surface tangent plane

Arrow meaning (important):

- Arrow **direction** follows the tangential field direction (`B_phi`, meridional)
- Arrow **length is normalized** (fixed plotting length), so it does **not** encode field magnitude
- Tangential field magnitude is encoded by the background colors
- The east-west arrow component is corrected by `cos(latitude)` so the direction is sensible on a lon/lat map


In [ ]:
#DONE did you sthink it was a good idea to hide this in a function?
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
img = ax.pcolormesh(
    lon_deg,
    lat_deg,
    b_tan,
    shading='nearest',
    cmap='viridis',
    norm='log',
)
img.cmap.set_under(img.cmap(0.0))

# Skip the polar rows for arrows (lon direction is singular at the poles).
I_STEP, J_STEP = 4, 6
lon_q = lon_deg[1:-1:I_STEP, ::J_STEP]
lat_q = lat_deg[1:-1:I_STEP, ::J_STEP]
b_phi_q = b_phi[1:-1:I_STEP, ::J_STEP]
b_mer_q = b_meridional[1:-1:I_STEP, ::J_STEP]

cos_lat = np.cos(np.deg2rad(lat_q))
u = b_phi_q / cos_lat
v = b_mer_q
arrow_len_deg = 8.0
mag = np.hypot(u, v)
u = arrow_len_deg * u / mag
v = arrow_len_deg * v / mag

ax.quiver(
    lon_q,
    lat_q,
    u,
    v,
    color='white',
    angles='xy',
    scale_units='xy',
    scale=1.0,
    width=0.0025,
    pivot='mid',
)

# Polarity inversion line: B_r = 0
ax.contour(lon_deg, lat_deg, b_r, levels=[0.0], colors='black', linewidths=0.8, alpha=0.7)

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.xaxis.set_major_locator(MultipleLocator(90))
ax.xaxis.set_minor_locator(MultipleLocator(30))
ax.yaxis.set_major_locator(MultipleLocator(45))
ax.yaxis.set_minor_locator(MultipleLocator(15))
ax.grid(which='major', alpha=0.15, linewidth=0.5)
ax.set_xlabel('Longitude [deg]')
ax.set_ylabel('Latitude [deg]')
ax.set_title(f'Tangential field directions at R={R_BOUNDARY:g} (background |B_tan|)')
fig.colorbar(img, ax=ax, label=f'|B_tan| [{B_PLOT_UNIT}]')
plt.show()


## Quick Boundary Summary (Useful Extra)

A few easy quantities to compute from the same shell sample:

- signed radial flux (should be small for a closed spherical surface in a divergence-free field)
- unsigned radial flux
- RMS strengths of radial / azimuthal / meridional / tangential components

In [ ]:
# Quick boundary summary from the same shell sample (no hidden summary helper)
signed_flux_Wb, signed_cov = integrate_shell_scalar(b_r_T[None, ...], shell.area[:1])
unsigned_flux_Wb, unsigned_cov = integrate_shell_scalar(np.abs(b_r_T)[None, ...], shell.area[:1])

summary = {
    'signed_radial_flux [Wb]': float(signed_flux_Wb[0]),
    'signed_flux_coverage [none]': float(signed_cov[0]),
    'unsigned_radial_flux [Wb]': float(unsigned_flux_Wb[0]),
    'unsigned_flux_coverage [none]': float(unsigned_cov[0]),
    f'RMS B_r [{B_PLOT_UNIT}]': float(np.sqrt(np.nanmean(b_r**2))),
    f'RMS B_azimuthal [{B_PLOT_UNIT}]': float(np.sqrt(np.nanmean(b_phi**2))),
    f'RMS B_meridional [{B_PLOT_UNIT}]': float(np.sqrt(np.nanmean(b_meridional**2))),
    f'RMS |B_tan| [{B_PLOT_UNIT}]': float(np.sqrt(np.nanmean(b_tan**2))),
}
summary


## Easy Follow-Ups

- compare two timesteps side-by-side using the same plotting helpers
- switch `background="radial"` in the vector plot to show tangential arrows over radial polarity
- repeat at `R > 1` to see how the magnetic topology opens with height